In [ ]:
# ===== Global KG construction configuration =====
# RUN_MODE options:
# - "visualize": run the single-sample KG demonstration; select v2 or nested_v1 with SINGLE_SAMPLE_EVENT_PROMPT_VERSION.
# - "postprocess": load the two cached CNC nested_v1 samples configured below and demonstrate post-processing; LM Studio is not required.
# - "rejudge": reuse saved extractions and rerun only the Judge and global diagnostics; LM Studio is not required.
# - "evaluation": rerun batch construction and evaluation with EVENT_PROMPT_VERSION.
RUN_MODE = "postprocess"

# Single-sample visualization settings (used when RUN_MODE="visualize")
MODE = "sample"                # "sample": select a dataset ID; "manual": use the text and spans below
DATASET = "cnc"                # cnc / li / ade / causenet
SAMPLE_ID = 5                  # Sample ID used when MODE="sample"
TRIPLE_SOURCE = "gold"         # "gold": use gold causal spans; "pred": run Demo1 first

# Manual mode: provide the text and causal spans directly, bypassing Demo1
MANUAL_TEXT = "5 Afghan troops killed after US army bombards warehouse in Kabul"
MANUAL_CAUSE = "US army bombards warehouse in Kabul"
MANUAL_EFFECT = "5 Afghan troops killed"

CAUSAL_PROMPT_VERSION = "v9.2"       # Prompt version for Demo1 causal extraction
SINGLE_SAMPLE_EVENT_PROMPT_VERSION = "v2"  # Single-sample option: "v2" or "nested_v1"; v2 remains the default
EVENT_PROMPT_VERSION = "nested_v1"          # Used when RUN_MODE="evaluation"
USE_RAG = False
RAG_MODE = "knn_pattern"           # "pattern", "knn", or "knn_pattern"
RAG_TOP_K = 3

# Two-example post-processing settings (used when RUN_MODE="postprocess"; this demonstration does not use v2)
POSTPROCESS_DATASET = "cnc"
POSTPROCESS_SAMPLE_IDS = [370, 371]  # Fixed demonstration with Kunming/Yunnan and factors reused across examples
POSTPROCESS_COLLECTION_ID = "cnc_demo_370_371"  # Label for this consolidation scope; source document IDs are not required
POSTPROCESS_EVENT_PROMPT_VERSION = "nested_v1"
POSTPROCESS_SOURCE_SPANS_PATH = "results/kg_evaluation/cnc_nested_v1_n300_20260901_132127_spans.jsonl"
POSTPROCESS_ENABLE_NER = True
STANZA_MODEL_DIR = None  # Set a custom directory if the models are not under ~/stanza_resources; section 2.6a loads them offline
STANZA_LANGUAGE = "en"
STANZA_PROCESSORS = "tokenize,ner"
STANZA_NER_PACKAGE = "ontonotes-ww-multi_charlm"
STANZA_USE_GPU = False  # CPU is sufficient for two demonstration samples; set True when CUDA is available
POSTPROCESS_ENABLE_WIKIPEDIA = True
POSTPROCESS_OUTPUT_DIR = "outputs/kg_postprocessing"
RDF_BASE_NAMESPACE = "https://example.org/master-thesis/causal-kg/"  # Prototype namespace; replace it for a formal publication
WIKIPEDIA_LANGUAGE = "en"
WIKIPEDIA_API_ENDPOINT = "https://en.wikipedia.org/w/api.php"
WIKIPEDIA_CACHE_PATH = "outputs/kg_postprocessing/wikipedia_link_cache.json"
WIKIPEDIA_TIMEOUT = 20
WIKIPEDIA_MAX_CANDIDATES = 5
WIKIPEDIA_FUZZY_THRESHOLD = 95.0  # Applies only when an exact-title page is unavailable
WIKIPEDIA_FUZZY_MARGIN = 5.0       # The leading candidate must exceed the runner-up by at least five points
WIKIPEDIA_USER_AGENT = "MasterThesisCausalKG/1.0 (academic research; local execution)"

# Evaluation mode: always use gold causal spans and evaluate one EVENT_PROMPT_VERSION at a time
EVAL_DATASET = "cnc"
EVAL_SAMPLE_N = 300                 # First N data points with gold causal relations; None selects all
EVAL_MAX_SPANS = None              # Optional total span limit; None disables the limit
EVAL_SCHEMA = "auto"               # "auto" / "two_layer" / "nested"
EVAL_MAX_DEPTH = "auto"            # "auto" / integer / None; two_layer=1, nested_depth3=3, nested_v1=None
EVAL_EXTRACTION_RETRY_TIMES = 1
EVAL_CHECKPOINT_EVERY = 100         # Save an intermediate checkpoint every 100 data points
RUN_JUDGE = True
JUDGE_PROMPT_VERSION = "v3"          # Keep v2 as a reference; v3 tightens semantic usefulness, scope, and attachment
JUDGE_API_KEY_PATH = "deepseek_api.txt"
JUDGE_MODEL = "deepseek-v4-pro"
JUDGE_MAX_TOKENS = 2048
JUDGE_TIMEOUT = 60
JUDGE_MAX_WORKERS = 10             # Concurrent DeepSeek judges; local construction extraction remains sequential
JUDGE_RETRY_TIMES = 3
JUDGE_RETRY_BASE_SECONDS = 2

# Rejudge mode: rerun only the Judge without connecting to LM Studio or repeating construction extraction
RUN_UNIT_REJUDGE = True
# Cached v2 extraction: results/kg_evaluation/cnc_v2_n300_20260901_060818_spans.jsonl
# Cached nested_v1 extraction: results/kg_evaluation/cnc_nested_v1_n300_20260901_132127_spans.jsonl
REJUDGE_SOURCE_SPANS_PATH = "results/kg_evaluation/cnc_nested_v1_n300_20260901_132127_spans.jsonl"
REJUDGE_OUTPUT_DIR = "results/kg_evaluation/rejudged"

# Independent global diagnostics: a second API pass that neither modifies nor feeds back into s/r/m/a
RUN_GLOBAL_DIAGNOSTICS = True
GLOBAL_DIAGNOSTIC_PROMPT_VERSION = "v1"
GLOBAL_DIAGNOSTIC_MAX_WORKERS = 10
GLOBAL_DIAGNOSTIC_OUTPUT_DIR = "results/kg_evaluation/global_diagnostics"

# LLM connection and generation settings
LLM_BASE_URL = "http://127.0.0.1:1234/v1"
LLM_API_KEY = "lm-studio"
MODEL_NAME = "auto"                # "auto" selects the single model currently loaded in LM Studio
TEMPERATURE = 0.0
MAX_TOKENS = 2048
CONTEXT_LENGTH = 8192
LLM_EXTRA_BODY = {}
LLM_TIMEOUT = 120
LLM_RETRY_TIMES = 3


In [ ]:
# ===== Initialization: environment, imports, client, and sample helpers =====
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Python executable: {sys.executable}")
if "master_thesis" not in sys.executable.lower():
    raise RuntimeError("This notebook is not using the Master_thesis kernel. Switch kernels and rerun it.")

from src.data_io import load_dataset
from src.event_extractor import build_kg_json
from src.generator import generate
from src.kg_eval_pipeline import (
    KGEvalConfig, KGRejudgeConfig, load_saved_span_results,
    resolve_eval_max_depth, resolve_eval_schema, run_kg_evaluation, run_saved_kg_judging,
)
from src.kg_global_diagnostics import (
    aggregate_global_diagnostics, prepare_global_diagnostic_records,
    run_parallel_global_diagnostics, save_global_diagnostic_outputs,
)
from src.kg_ner_enrichment import enrich_collection_with_ner
from src.kg_postprocess_visualizer import POSTPROCESS_LEGEND_HTML, build_postprocess_graphs
from src.kg_postprocessor import consolidate_kg_collection, deduplicate_kg_samples, load_cached_kg_samples, normalize_kg_samples
from src.kg_wikipedia_linker import link_collection_to_wikipedia
from src.kg_serializer import serialize_integrated_kg
from src.kg_visualizer import visualize
from src.kg_evaluator import (
    DeepSeekJudgeClient,
    aggregate_sample_results,
    build_judge_prompt,
    compute_span_metrics,
    flatten_judge_units,
    load_gold_span_records,
    parse_judge_output,
    run_parallel_judge,
    run_eval_extraction,
    save_eval_checkpoint,
    save_eval_outputs,
    validate_extraction,
)
from src.llm_client import LLMClient
from src.retriever import create_retriever

if RUN_MODE in {"rejudge", "postprocess"}:
    client = None
    retriever = None
    print(f"{RUN_MODE} mode: skip LM Studio connection and reuse saved parsed extractions.")
else:
    client = LLMClient(
        base_url=LLM_BASE_URL,
        model=MODEL_NAME,
        api_key=LLM_API_KEY,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
        context_length=CONTEXT_LENGTH,
        extra_body=LLM_EXTRA_BODY,
        timeout=LLM_TIMEOUT,
        retry_times=LLM_RETRY_TIMES,
    )
    if MODEL_NAME == "auto":
        loaded_models = client.list_loaded_models()
        if len(loaded_models) == 1:
            client.model = loaded_models[0]
            model_source = "LM Studio auto"
        elif len(loaded_models) == 0:
            raise RuntimeError("MODEL_NAME='auto', but LM Studio has no loaded chat model.")
        else:
            raise RuntimeError(f"MODEL_NAME='auto', but LM Studio returned multiple models: {loaded_models}. Set MODEL_NAME explicitly.")
    else:
        model_source = "notebook MODEL_NAME"
    retriever = create_retriever(RAG_MODE) if USE_RAG else None
    print(f"Base URL: {client.base_url}")
    print(f"Model: {client.model}")
    print(f"Model source: {model_source}")
print(f"Triple source: {TRIPLE_SOURCE}")
print(f"RUN_MODE: {RUN_MODE}")


def pred_triples_to_kg_triples(prediction):
    if not prediction.get("has_causal", False):
        return []
    return prediction.get("triples", []) if isinstance(prediction.get("triples"), list) else []


def gold_relations_to_kg_triples(sample):
    if not sample.get("has_causal", False):
        return []
    triples = []
    for relation in sample.get("relations", []):
        triples.append({"cause": relation.get("cause", ""), "effect": relation.get("effect", ""), "relation": "caused"})
    return triples


def load_sample_by_id(dataset, sample_id):
    samples = load_dataset(dataset)
    for sample in samples:
        if sample.get("id") == sample_id:
            return sample
    raise ValueError(f"No sample with id={sample_id} was found in {dataset}")


def build_current_kg_json():
    if MODE == "manual":
        triples = [{"cause": MANUAL_CAUSE, "effect": MANUAL_EFFECT, "relation": "caused"}]
        return build_kg_json(
            sample_id=None,
            text=MANUAL_TEXT,
            triples=triples,
            client=client,
            prompt_version=SINGLE_SAMPLE_EVENT_PROMPT_VERSION,
            triple_source="manual",
        )

    sample = load_sample_by_id(DATASET, SAMPLE_ID)
    if TRIPLE_SOURCE == "gold":
        triples = gold_relations_to_kg_triples(sample)
        print("Gold relations:")
        print(json.dumps(sample.get("relations", []), indent=2, ensure_ascii=False))
    elif TRIPLE_SOURCE == "pred":
        prediction_key = (DATASET, SAMPLE_ID, CAUSAL_PROMPT_VERSION, USE_RAG, RAG_MODE, RAG_TOP_K)
        if globals().get("demo1_prediction_key") == prediction_key and "demo1_prediction" in globals():
            prediction = demo1_prediction
            print("Using the predicted triples from the preceding Demo1 preview cell.")
        else:
            prediction = generate(
                text=sample["text"],
                sample_id=sample["id"],
                client=client,
                retriever=retriever,
                use_rag=USE_RAG,
                top_k=RAG_TOP_K,
                rag_mode=RAG_MODE,
                prompt_name=CAUSAL_PROMPT_VERSION,
            )
        triples = pred_triples_to_kg_triples(prediction)
        print("Gold relations:")
        print(json.dumps(sample.get("relations", []), indent=2, ensure_ascii=False))
        print("Pred triples:")
        print(json.dumps(prediction.get("triples", []), indent=2, ensure_ascii=False))
    else:
        raise ValueError("TRIPLE_SOURCE must be either 'gold' or 'pred'")

    if not triples:
        print(f"The current sample has no {TRIPLE_SOURCE} causal triple from which to build a graph.")
    return build_kg_json(
        sample_id=sample["id"],
        text=sample["text"],
        triples=triples,
        client=client,
        prompt_version=SINGLE_SAMPLE_EVENT_PROMPT_VERSION,
        triple_source=TRIPLE_SOURCE,
    )


## 1. Single-sample KG construction demonstration (v2 or nested_v1)

This workflow retains the original single-sample demonstration. With `RUN_MODE="visualize"`, Demo1 supplies the causal triples, and `SINGLE_SAMPLE_EVENT_PROMPT_VERSION` selects either `"v2"` or `"nested_v1"` for the internal event structure.


In [ ]:
# ===== Demo1 extraction and evaluation preview for the current sample =====
if RUN_MODE != "visualize":
    print("RUN_MODE!='visualize'; skipping this single-sample demonstration cell.")
else:
    # Inspect Demo1 causal extraction and single-sample evaluation before continuing to KG construction.
    from src.evaluator import build_sample_judgement

    if MODE == "manual":
        selected_sample = {
            "id": None,
            "text": MANUAL_TEXT,
            "has_causal": True,
            "relations": [{"cause": MANUAL_CAUSE, "effect": MANUAL_EFFECT}],
        }
        demo1_prediction = {
            "id": None,
            "has_causal": True,
            "triples": [{"cause": {"span": MANUAL_CAUSE}, "relation": "caused", "effect": {"span": MANUAL_EFFECT}}],
        }
        demo1_prediction_key = None
        print("MODE='manual': using the supplied cause and effect spans for the Demo1 preview.")
    else:
        selected_sample = load_sample_by_id(DATASET, SAMPLE_ID)
        demo1_prediction_key = (DATASET, SAMPLE_ID, CAUSAL_PROMPT_VERSION, USE_RAG, RAG_MODE, RAG_TOP_K)
        demo1_prediction = generate(
            text=selected_sample["text"],
            sample_id=selected_sample["id"],
            client=client,
            retriever=retriever,
            use_rag=USE_RAG,
            top_k=RAG_TOP_K,
            rag_mode=RAG_MODE,
            prompt_name=CAUSAL_PROMPT_VERSION,
        )

    demo1_judgement = build_sample_judgement(
        prediction=demo1_prediction,
        gold=selected_sample,
        dataset=DATASET if MODE == "sample" else None,
    )

    print(f"Sample id: {selected_sample.get('id')}")
    print(f"Text: {selected_sample['text']}")
    print("Gold relations:")
    print(json.dumps(selected_sample.get("relations", []), indent=2, ensure_ascii=False))
    print("Pred triples:")
    print(json.dumps(demo1_prediction.get("triples", []), indent=2, ensure_ascii=False))
    print("Single-sample judgement:")
    print(json.dumps({
        "gold_has_causal": demo1_judgement["gold_has_causal"],
        "pred_has_causal": demo1_judgement["pred_has_causal"],
        "primary_metric": demo1_judgement["primary_metric"],
        "strict_token_f1_counts": demo1_judgement["strict_token_f1"]["counts"],
        "anchor_window_counts": demo1_judgement["anchor_window"]["counts"],
    }, indent=2, ensure_ascii=False))


In [ ]:
# ===== Single-sample construction (v2 or nested_v1): build the KG JSON =====
if RUN_MODE != "visualize":
    print("RUN_MODE!='visualize'; skipping this single-sample demonstration cell.")
else:
    # - MODE="sample" loads the selected dataset sample and chooses gold or predicted triples through TRIPLE_SOURCE.
    # - If the selected source has no graphable triple, empty causal_links and events are expected.
    current_kg = build_current_kg_json()
    print(json.dumps(current_kg, indent=2, ensure_ascii=False))
    print(f"causal_links: {len(current_kg['causal_links'])}")
    print(f"events: {len(current_kg['events'])}")


In [ ]:
# ===== Single-sample construction (v2 or nested_v1): KG JSON to NetworkX =====
if RUN_MODE != "visualize":
    print("RUN_MODE!='visualize'; skipping this single-sample demonstration cell.")
else:
    import networkx as nx
    from src.kg_builder import build_graph
    from src.kg_visualizer import visualize

    # Build a NetworkX graph from the single-sample KG JSON produced by the selected prompt.
    if "current_kg" not in globals():
        current_kg = build_current_kg_json()

    G = build_graph(current_kg)
    print(f"Number of nodes: {G.number_of_nodes()}")
    print(f"Number of edges: {G.number_of_edges()}")

    print("\nNodes:")
    for node_id, data in G.nodes(data=True):
        role = data.get("role") or data.get("event_role")
        print(f"- {node_id}: type={data.get('type')}, role={role}, label={data.get('label')}")

    print("\nEdges:")
    for source, target, data in G.edges(data=True):
        print(f"- {source} -> {target}: {data.get('label')}")


In [ ]:
# ===== Single-sample construction (v2 or nested_v1): interactive pyvis HTML =====
import os
if RUN_MODE != "visualize":
    print("RUN_MODE!='visualize'; skipping this single-sample demonstration cell.")
else:
    from pathlib import Path
    from IPython.display import IFrame

    # Render the single-sample KG produced by the selected prompt as interactive HTML.
    if "current_kg" not in globals():
        current_kg = build_current_kg_json()
    if "G" not in globals():
        G = build_graph(current_kg)

    source_label = "manual" if MODE == "manual" else f"{DATASET}_{SAMPLE_ID}_{TRIPLE_SOURCE}"
    html_path = visualize(G, str(PROJECT_ROOT / "outputs" / "kg_html" / f"{source_label}.html"))
    print(f"Visualization written to: {html_path}")

    display_path = Path(html_path).resolve()
    try:
        iframe_src = Path(os.path.relpath(display_path, Path.cwd().resolve())).as_posix()
    except ValueError:
        iframe_src = display_path.as_uri()

    IFrame(iframe_src, width=900, height=600)


## 2. Post-generation processing demonstration (nested_v1)

This section uses the two configured CNC `nested_v1` construction results (samples 370 and 371 by default). Causal triples and their event/span nodes remain distinct throughout. Post-processing adds normalization, deterministic boundary-function-word deduplication, cross-example canonical-resource consolidation, provenance, NER, Wikipedia linking, and serialization. The demonstration does not use v2.


In [ ]:
# ===== 2.1 Load or rebuild the nested_v1 KGs for the two configured CNC samples =====
if RUN_MODE != "postprocess":
    print("RUN_MODE!='postprocess'; skipping the two-example KG loading step.")
else:
    postprocess_source = Path(POSTPROCESS_SOURCE_SPANS_PATH)
    if not postprocess_source.is_absolute():
        postprocess_source = PROJECT_ROOT / postprocess_source

    raw_demo_kgs = load_cached_kg_samples(
        postprocess_source,
        dataset=POSTPROCESS_DATASET,
        sample_ids=POSTPROCESS_SAMPLE_IDS,
        prompt_version=POSTPROCESS_EVENT_PROMPT_VERSION,
    )

    print(f"Loaded cached construction: {postprocess_source}")
    for sample_id, kg_json in raw_demo_kgs.items():
        print(
            f"sample={sample_id}: events={len(kg_json['events'])}, "
            f"causal_links={len(kg_json['causal_links'])}, "
            f"prompt={kg_json['construction_prompt_version']}"
        )


In [ ]:
# ===== 2.2 Display the two unprocessed nested_v1 KGs =====
if RUN_MODE != "postprocess":
    print("RUN_MODE!='postprocess'; skipping the raw KG preview.")
else:
    if "raw_demo_kgs" not in globals():
        raise RuntimeError("Run section 2.1 before this cell.")

    for sample_id in POSTPROCESS_SAMPLE_IDS:
        print(f"\n{'=' * 24} CNC {sample_id} raw nested_v1 KG {'=' * 24}")
        print(json.dumps(raw_demo_kgs[sample_id], indent=2, ensure_ascii=False))


In [ ]:
# ===== 2.3 Deterministic normalization =====
# Deterministically standardize Unicode, quotation marks, hyphens, whitespace, and case-folded keys.
# Do not stem, merge synonyms, or disambiguate entities; always preserve the original value.
if RUN_MODE != "postprocess":
    print("RUN_MODE!='postprocess'; skipping deterministic normalization.")
else:
    if "raw_demo_kgs" not in globals():
        raise RuntimeError("Run section 2.1 before this cell.")

    normalized_demo_kgs = normalize_kg_samples(raw_demo_kgs)

    def collect_normalized_rows(sample_id, kg_json):
        rows = []

        def visit(unit, event_id, path):
            rows.append({
                "sample_id": sample_id,
                "event_id": event_id,
                "path": path,
                "role": unit["role"],
                "value": unit["value"],
                "normalized_value": unit["normalized_value"],
                "canonical_key": unit["canonical_key"],
            })
            for index, attribute in enumerate(unit.get("attributes", [])):
                visit(attribute, event_id, f"{path}.attributes[{index}]")
            for index, child in enumerate(unit.get("children", [])):
                visit(child, event_id, f"{path}.children[{index}]")

        for event_id, event in kg_json["events"].items():
            for index, component in enumerate(event["components"]):
                visit(component, event_id, f"components[{index}]")
        return rows

    normalized_rows = []
    for sample_id, kg_json in normalized_demo_kgs.items():
        sample_rows = collect_normalized_rows(sample_id, kg_json)
        normalized_rows.extend(sample_rows)
        print(f"\nCNC {sample_id} normalization metadata:")
        print(json.dumps(kg_json["normalization"], indent=2, ensure_ascii=False))
        print(json.dumps(sample_rows, indent=2, ensure_ascii=False))

    keys_by_sample = {
        sample_id: {row["canonical_key"] for row in collect_normalized_rows(sample_id, kg_json)}
        for sample_id, kg_json in normalized_demo_kgs.items()
    }
    shared_keys = sorted(set.intersection(*(set(keys) for keys in keys_by_sample.values())))
    print("\nShared exact canonical keys across the two examples:")
    print(json.dumps(shared_keys, indent=2, ensure_ascii=False))


In [ ]:
# ===== 2.4 Within-example deduplication =====
# General rule: apply exact matching first; after stripping boundary function words, merge only when the target strict key already exists in the sample.
# Do not remove phrase-internal function words or use a model; retain every original mention, event, causal link, and matching rationale.
if RUN_MODE != "postprocess":
    print("RUN_MODE!='postprocess'; skipping within-example deduplication.")
else:
    if "normalized_demo_kgs" not in globals():
        raise RuntimeError("Run section 2.3 before this cell.")

    deduplicated_demo_kgs = deduplicate_kg_samples(normalized_demo_kgs)

    for sample_id, kg_json in deduplicated_demo_kgs.items():
        summary = kg_json["within_example_deduplication"]
        duplicate_groups = [
            resource
            for resource in kg_json["canonical_resources"]
            if resource["mention_count"] > 1
        ]
        print(f"\nCNC {sample_id} within-example deduplication summary:")
        print(json.dumps(summary, indent=2, ensure_ascii=False))
        if duplicate_groups:
            print("Shared canonical resources in this example:")
            print(json.dumps(duplicate_groups, indent=2, ensure_ascii=False))
        else:
            print("No exact or conditional boundary match within this example.")


In [ ]:
# ===== 2.5 Collection-level canonical resource consolidation + provenance =====
# General rule: apply exact matching first; after stripping boundary function words, merge only when the target strict key already exists in the collection.
# Use the same deterministic rule as section 2.4. This creates surface-form clusters without claiming entity disambiguation or identity resolution.
if RUN_MODE != "postprocess":
    print("RUN_MODE!='postprocess'; skipping collection-level consolidation.")
else:
    if "deduplicated_demo_kgs" not in globals():
        raise RuntimeError("Run section 2.4 before this cell.")

    integrated_kg = consolidate_kg_collection(
        deduplicated_demo_kgs,
        collection_id=POSTPROCESS_COLLECTION_ID,
    )
    print("Collection consolidation summary:")
    print(json.dumps(integrated_kg["collection_consolidation"], indent=2, ensure_ascii=False))

    cross_example_resources = [
        resource
        for resource in integrated_kg["canonical_resources"]
        if resource["sample_count"] > 1
    ]
    print("\nCanonical resources shared across examples:")
    print(json.dumps(cross_example_resources, indent=2, ensure_ascii=False))


In [ ]:
# ===== 2.6a Stanza NER mention detection + existing-node reconciliation =====
# For an exact match to an existing factor, or an entity already assigned to the same canonical resource in sections 2.4/2.5, add only ner_annotations.
# For any other strict subspan without an existing factor or resource, create a separate ner_mention linked to its source factor through has_ner_mention.
# This cell does not access Wikipedia.
if RUN_MODE != "postprocess":
    print("RUN_MODE!='postprocess'; skipping NER enrichment.")
else:
    if "integrated_kg" not in globals():
        raise RuntimeError("Run section 2.5 before this cell.")
    if not POSTPROCESS_ENABLE_NER:
        ner_enriched_kg = integrated_kg
        print("POSTPROCESS_ENABLE_NER=False; retaining integrated_kg without NER enrichment.")
    else:

        import stanza

        stanza_model_dir = Path(STANZA_MODEL_DIR).expanduser() if STANZA_MODEL_DIR else Path.home() / "stanza_resources"
        if not stanza_model_dir.exists():
            raise FileNotFoundError(f"Stanza model directory not found: {stanza_model_dir}")
        stanza_pipeline_key = (
            str(stanza_model_dir.resolve()),
            STANZA_LANGUAGE,
            STANZA_PROCESSORS,
            STANZA_NER_PACKAGE,
            STANZA_USE_GPU,
        )
        if (
            globals().get("_stanza_pipeline_key") != stanza_pipeline_key
            or "stanza_ner_pipeline" not in globals()
        ):
            stanza_ner_pipeline = stanza.Pipeline(
                lang=STANZA_LANGUAGE,
                processors=STANZA_PROCESSORS,
                package={"ner": STANZA_NER_PACKAGE},
                dir=str(stanza_model_dir),
                use_gpu=STANZA_USE_GPU,
                download_method=None,
                verbose=False,
            )
            _stanza_pipeline_key = stanza_pipeline_key
            print(f"Loaded Stanza NER from {stanza_model_dir.resolve()}")
        else:
            print("Reuse loaded Stanza NER pipeline.")

        ner_enriched_kg = enrich_collection_with_ner(
            integrated_kg,
            ner_pipeline=stanza_ner_pipeline,
            detector_name="stanza",
            detector_model=f"{STANZA_LANGUAGE}/{STANZA_NER_PACKAGE}",
            detector_version=stanza.__version__,
        )
        print("NER enrichment summary:")
        print(json.dumps(ner_enriched_kg["ner_enrichment"], indent=2, ensure_ascii=False))
        print("\nNER detections and reconciliation actions:")
        print(json.dumps(ner_enriched_kg["ner_detections"], indent=2, ensure_ascii=False))
        print("\nNew NER mention nodes:")
        print(json.dumps(ner_enriched_kg["ner_mentions"], indent=2, ensure_ascii=False))


In [ ]:
# ===== 2.6b Wikipedia entity/concept linking =====
# Check every canonical resource against English Wikipedia; NER status, role, and leaf status are not filters.
# Link exact-title pages and redirects directly, mark disambiguation or low-confidence candidates as ambiguous, and record network failures as request_failed.
import pandas as pd
if RUN_MODE != "postprocess":
    print("RUN_MODE!='postprocess'; skipping Wikipedia linking.")
else:
    if "ner_enriched_kg" not in globals():
        raise RuntimeError("Run section 2.6a before this cell.")
    if not POSTPROCESS_ENABLE_WIKIPEDIA:
        linked_integrated_kg = ner_enriched_kg
        print("POSTPROCESS_ENABLE_WIKIPEDIA=False; retaining ner_enriched_kg without Wikipedia links.")
    else:
        wikipedia_cache_path = Path(WIKIPEDIA_CACHE_PATH)
        if not wikipedia_cache_path.is_absolute():
            wikipedia_cache_path = PROJECT_ROOT / wikipedia_cache_path

        linked_integrated_kg = link_collection_to_wikipedia(
            ner_enriched_kg,
            cache_path=wikipedia_cache_path,
            endpoint=WIKIPEDIA_API_ENDPOINT,
            language=WIKIPEDIA_LANGUAGE,
            timeout=WIKIPEDIA_TIMEOUT,
            max_candidates=WIKIPEDIA_MAX_CANDIDATES,
            fuzzy_threshold=WIKIPEDIA_FUZZY_THRESHOLD,
            fuzzy_margin=WIKIPEDIA_FUZZY_MARGIN,
            user_agent=WIKIPEDIA_USER_AGENT,
        )
        print("Wikipedia linking summary:")
        print(json.dumps(linked_integrated_kg["wikipedia_linking"], indent=2, ensure_ascii=False))

        wikipedia_rows = []
        for resource in linked_integrated_kg["canonical_resources"]:
            link = resource["wikipedia_link"]
            selected = link.get("selected_page") or {}
            wikipedia_rows.append({
                "canonical_key": resource["canonical_key"],
                "surface_forms": " | ".join(resource.get("surface_forms", [])),
                "status": link["status"],
                "match_method": link.get("match_method"),
                "wikipedia_title": selected.get("title"),
                "wikipedia_url": selected.get("url"),
                "cache_hit": link.get("cache_hit", False),
            })
        display(pd.DataFrame(wikipedia_rows))


In [ ]:
# ===== 2.7 NetworkX and pyvis visualization before and after consolidation =====
# Raw graphs retain the nested_v1 event/factor structure; the integrated graph collapses each canonical resource into one node.
# Original mentions are not drawn as duplicate nodes, but their source, role, path, and related provenance remain on nodes and edges.
# Visual encodings are independent: diamonds marked [SHARED] denote cross-example nodes, while gold nodes with green borders marked [WIKI] denote successful Wikipedia links.
# A node may therefore be both diamond-shaped and gold without conflating deduplication with external linking.
if RUN_MODE != "postprocess":
    print("RUN_MODE!='postprocess'; skipping post-processing visualization.")
else:
    if "linked_integrated_kg" not in globals():
        raise RuntimeError("Run section 2.6b before this cell.")

    raw_demo_graphs, integrated_graph = build_postprocess_graphs(
        raw_demo_kgs,
        linked_integrated_kg,
    )

    postprocess_output_dir = Path(POSTPROCESS_OUTPUT_DIR)
    if not postprocess_output_dir.is_absolute():
        postprocess_output_dir = PROJECT_ROOT / postprocess_output_dir
    postprocess_output_dir.mkdir(parents=True, exist_ok=True)

    raw_demo_html_paths = {}
    for sample_id, graph in raw_demo_graphs.items():
        path = postprocess_output_dir / f"{POSTPROCESS_COLLECTION_ID}_raw_{sample_id}.html"
        raw_demo_html_paths[sample_id] = Path(visualize(graph, str(path), height="520px"))

    postprocess_html_path = Path(visualize(
        integrated_graph,
        str(postprocess_output_dir / f"{POSTPROCESS_COLLECTION_ID}_integrated.html"),
        height="780px",
        legend_html=POSTPROCESS_LEGEND_HTML,
    ))

    shared_resources = [
        data for _, data in integrated_graph.nodes(data=True)
        if data.get("type") == "canonical_resource" and data.get("is_cross_example")
    ]
    wiki_resources = [
        data for _, data in integrated_graph.nodes(data=True)
        if data.get("type") == "canonical_resource" and data.get("wikipedia_linked")
    ]
    visualization_summary = {
        "raw_graph_count": len(raw_demo_graphs),
        "integrated_node_count": integrated_graph.number_of_nodes(),
        "integrated_edge_count": integrated_graph.number_of_edges(),
        "shared_canonical_nodes": sorted(item["preferred_label"] for item in shared_resources),
        "wikipedia_linked_nodes": sorted(item["preferred_label"] for item in wiki_resources),
        "raw_html_paths": {key: str(value) for key, value in raw_demo_html_paths.items()},
        "integrated_html_path": str(postprocess_html_path),
    }
    print(json.dumps(visualization_summary, indent=2, ensure_ascii=False))

    # Use Jupyter's /files/ route instead of a file:// URL that the localhost page may block.
    from urllib.parse import quote
    from IPython.display import IFrame, display

    def notebook_file_url(html_path):
        relative_path = Path(html_path).resolve().relative_to(PROJECT_ROOT.resolve()).as_posix()
        return f"/files/{quote(relative_path, safe='/')}"

    # Show the two pre-consolidation graphs first, followed by the final graph with canonical resources collapsed.
    for sample_id in POSTPROCESS_SAMPLE_IDS:
        print(f"Raw nested_v1 KG — CNC {sample_id}")
        iframe_url = notebook_file_url(raw_demo_html_paths[sample_id])
        print(f"Notebook preview URL: {iframe_url}")
        display(IFrame(src=iframe_url, width="100%", height=560))
    print("Integrated post-processed KG")
    iframe_url = notebook_file_url(postprocess_html_path)
    print(f"Notebook preview URL: {iframe_url}")
    display(IFrame(src=iframe_url, width="100%", height=860))


In [ ]:
# ===== 2.8 Turtle and JSON-LD serialization =====
# Common roles explicitly defined by the prompt use fixed predicates; freely generated LLM roles use stable custom predicates.
# Each custom predicate retains the original role label and is declared as a sub-property of ex:hasFactorRelation rather than being discarded or forced into another role.
# The Wikipedia pilot uses rdfs:seeAlso instead of the semantically stronger owl:sameAs.
from src.kg_serializer import DEFAULT_BASE_NAMESPACE, serialize_integrated_kg
if RUN_MODE != "postprocess":
    print("RUN_MODE!='postprocess'; skipping RDF serialization.")
else:
    if "linked_integrated_kg" not in globals():
        raise RuntimeError("Run section 2.6b before this cell.")

    rdf_output_dir = Path(POSTPROCESS_OUTPUT_DIR)
    if not rdf_output_dir.is_absolute():
        rdf_output_dir = PROJECT_ROOT / rdf_output_dir
    turtle_path = rdf_output_dir / f"{POSTPROCESS_COLLECTION_ID}.ttl"
    jsonld_path = rdf_output_dir / f"{POSTPROCESS_COLLECTION_ID}.jsonld"

    rdf_serialization_summary = serialize_integrated_kg(
        linked_integrated_kg,
        turtle_path=turtle_path,
        jsonld_path=jsonld_path,
        base_namespace=globals().get("RDF_BASE_NAMESPACE", DEFAULT_BASE_NAMESPACE),
    )
    print("RDF serialization and round-trip validation:")
    print(json.dumps(rdf_serialization_summary, indent=2, ensure_ascii=False))
    if rdf_serialization_summary["used_open_roles"]:
        print("\nLLM-generated open roles retained as custom predicates:")
        print(json.dumps(rdf_serialization_summary["used_open_roles"], indent=2, ensure_ascii=False))


## 3. KG evaluation and diagnostics

The following workflows are independent of the post-processing demonstration: `rejudge` reuses saved extractions, while `evaluation` uses gold causal spans for batch construction and evaluation.


In [ ]:
# ===== KG rejudging: reuse saved extractions and run only the revised Judge =====
# The default input is the cached 300-sample nested_v1 run; LM Studio is not required.
if RUN_MODE != "rejudge" or not RUN_UNIT_REJUDGE:
    print("Rejudge/unit Judge is disabled; skipping cached-unit re-evaluation.")
else:
    try:
        from tqdm.auto import tqdm as rejudge_tqdm
    except Exception:
        rejudge_tqdm = None

    rejudge_source = Path(REJUDGE_SOURCE_SPANS_PATH)
    if not rejudge_source.is_absolute():
        rejudge_source = PROJECT_ROOT / rejudge_source
    if not rejudge_source.exists():
        raise FileNotFoundError(f"Cached span file not found: {rejudge_source}")

    print(f"Cached extraction: {rejudge_source}")
    print(f"Judge: prompt={JUDGE_PROMPT_VERSION}; model={JUDGE_MODEL}; workers={JUDGE_MAX_WORKERS}")
    print("LM Studio / construction extraction: skipped")

    rejudge_config = KGRejudgeConfig(
        source_spans_path=rejudge_source,
        judge_prompt_version=JUDGE_PROMPT_VERSION,
        schema=EVAL_SCHEMA,
        max_depth=EVAL_MAX_DEPTH,
        output_dir=PROJECT_ROOT / REJUDGE_OUTPUT_DIR,
        judge_api_key_path=PROJECT_ROOT / JUDGE_API_KEY_PATH,
        judge_model=JUDGE_MODEL,
        judge_max_tokens=JUDGE_MAX_TOKENS,
        judge_timeout=JUDGE_TIMEOUT,
        judge_max_workers=JUDGE_MAX_WORKERS,
        judge_retry_times=JUDGE_RETRY_TIMES,
        judge_retry_base_seconds=JUDGE_RETRY_BASE_SECONDS,
        save_outputs=True,
    )
    rejudge_result = run_saved_kg_judging(rejudge_config, progress_factory=rejudge_tqdm)

    span_results = rejudge_result.span_results
    sample_results = rejudge_result.sample_results
    method_metrics = rejudge_result.method_metrics
    output_paths = rejudge_result.output_paths
    print("\n===== KG Rejudge Summary =====")
    for key, value in method_metrics.items():
        print(f"{key}: {value}")
    print("\nSaved files:")
    for name, path in output_paths.items():
        print(f"{name}: {path}")


In [ ]:
# ===== KG global diagnostics: independent full-tree checks outside the four Judge-derived metrics =====
# Make a second, independent DeepSeek call over the same cached extractions.
if RUN_MODE != "rejudge" or not RUN_GLOBAL_DIAGNOSTICS:
    print("Rejudge/global diagnostics is disabled; skipping global diagnostics.")
else:
    try:
        from tqdm.auto import tqdm as global_tqdm
    except Exception:
        global_tqdm = None

    global_source = Path(REJUDGE_SOURCE_SPANS_PATH)
    if not global_source.is_absolute():
        global_source = PROJECT_ROOT / global_source
    cached_span_results = load_saved_span_results(global_source)
    global_records = prepare_global_diagnostic_records(
        cached_span_results,
        prompt_version=GLOBAL_DIAGNOSTIC_PROMPT_VERSION,
    )

    global_client = DeepSeekJudgeClient(
        api_key_path=PROJECT_ROOT / JUDGE_API_KEY_PATH,
        model=JUDGE_MODEL,
        max_tokens=JUDGE_MAX_TOKENS,
        timeout=JUDGE_TIMEOUT,
        retry_times=JUDGE_RETRY_TIMES,
        retry_base_seconds=JUDGE_RETRY_BASE_SECONDS,
    )
    global_progress = global_tqdm(total=len(global_records), desc="Global diagnostic spans") if global_tqdm else None
    try:
        global_records = run_parallel_global_diagnostics(
            global_records,
            global_client,
            max_workers=GLOBAL_DIAGNOSTIC_MAX_WORKERS,
            progress_callback=(lambda _record: global_progress.update(1)) if global_progress is not None else None,
        )
    finally:
        if global_progress is not None:
            global_progress.close()

    global_metrics = aggregate_global_diagnostics(global_records)
    global_dataset = str(cached_span_results[0].get("dataset", "unknown"))
    global_event_prompt = str(cached_span_results[0].get("prompt_version", "unknown"))
    global_sample_count = len({(item.get("dataset"), item.get("sample_id")) for item in cached_span_results})
    global_output_paths = save_global_diagnostic_outputs(
        global_records,
        global_metrics,
        output_dir=PROJECT_ROOT / GLOBAL_DIAGNOSTIC_OUTPUT_DIR,
        dataset=global_dataset,
        event_prompt_version=global_event_prompt,
        sample_count=global_sample_count,
        prompt_version=GLOBAL_DIAGNOSTIC_PROMPT_VERSION,
    )

    print("\n===== Independent Global Diagnostics =====")
    for key, value in global_metrics.items():
        display_value = "N/A" if value is None else value
        print(f"{key}: {display_value}")
    print("\nSaved global diagnostic files:")
    for name, path in global_output_paths.items():
        print(f"{name}: {path}")


## KG evaluation metrics

This evaluation section assesses KG construction only; it does not reevaluate causal extraction in Demo1. The inputs are fixed gold-standard cause and effect spans. For each span, the local construction model first extracts an event or state structure, which is then checked by deterministic validation and optionally assessed by the DeepSeek semantic Judge.

### Deterministic metrics

These metrics are calculated directly in code and do not depend on the LLM Judge. They check output validity, exact-substring constraints, structural depth, and output size.

- `json_parse_success`: whether the construction output can be parsed as a JSON object. `1` indicates success and `0` indicates failure.
- `schema_valid`: whether the parsed output follows the expected schema, including a `components` list and non-empty `role` and `value` fields for nodes and attributes.
- `substring_valid`: whether every node or attribute `value` is an exact contiguous substring of the input span. Lemmatization, normalization, and paraphrasing do not satisfy this strict check.
- `depth_compliant`: whether structural nesting remains within `EVAL_MAX_DEPTH`. By default, `nested_v1` has no depth limit, `two_layer` has a maximum depth of 1, and `nested_depth3` has a maximum depth of 3.
- `forbidden_role_count`: the number of prohibited roles, such as `Cause`, `Effect`, `Reason`, or `Consequence`. KG construction should not relabel the outer causal relation.
- Span-level `max_depth`: the deepest node level reached by one span output. Top-level components have depth 1, child nodes depth 2, and grandchildren depth 3; attributes do not increase node depth. The aggregate mean is reported as `avg_max_depth`, while the deepest observed structure is reported as `max_observed_depth`.
- `avg_max_depth`: the aggregate mean of span-level maximum depth, describing the average nesting depth produced by the method.
- `max_observed_depth`: the maximum depth observed across all evaluated outputs. Use this field to identify the deepest structure produced by the model.
- `node_count`: the number of node units, including top-level components and nested child nodes.
- `attribute_count`: the number of attribute units.
- `child_link_count`: the number of parent-child links between nested nodes.
- `judge_unit_count`: the number of units sent to the semantic Judge, including nodes, attributes, and child links.
- `deterministic_compliance_rate`: the proportion for which `json_parse_success`, `schema_valid`, and `substring_valid` all pass. The main table reports this combined rate, while the three underlying fields remain in the full results.
- `forbidden_role_affected_rate`: the proportion of spans containing at least one prohibited role. The raw `forbidden_role_count` is also retained.
- `content_token_coverage`: the proportion of non-punctuation, non-fixed-function-word tokens in the gold span covered by at least one node or attribute value. This measures lexical coverage, not semantic recall.
- `exact_duplicate_rate`, `containment_rate`, and `normalized_chain_depth`: retained audit and screening metrics that are omitted from the compact main table. In a nested architecture, containment cannot be interpreted directly as redundancy.
- `p95_max_depth`: the 95th percentile of span-level maximum depth, reported alongside the mean to expose long-tail nesting.

The outputs contain both `_metrics.csv` with all metrics and `_main_metrics.csv` with the compact main-table fields. Metrics are calculated and retained in full while the main table remains concise.

### Purpose of the LLM Judge

The DeepSeek Judge is a precision-oriented semantic evaluator. It does not determine whether the model extracted everything that should have been extracted. Instead, it assesses whether the generated units are semantically grounded, correctly labelled, appropriately scoped, and attached to reasonable parent nodes.

The Judge is intentionally separated from deterministic validation. It does not assess JSON validity, schema validity, substring validity, depth compliance, or counts because these are checked in code. It also does not penalize omitted information, so its scores primarily measure precision-like quality rather than recall.

The Judge receives units flattened from one source span:

- `node`: a component or nested child node, such as Action, Actor, Theme, Location, or Time.
- `attribute`: modifying information attached to a node, such as Quantifier, Negation, Age, Name, Type, Topic, Time, or Location.
- `child_link`: a parent-child relation between two nested nodes.

For each unit, the Judge may assign only `1`, `0.5`, `0`, or `null`. A score of `1` is correct, `0.5` is partially correct or acceptable but imperfect, `0` is incorrect, and `null` means that the field does not apply to that unit type.

### LLM Judge fields

The Judge returns the following raw fields for each unit.

- `s` / semantic usefulness (v3): whether the unit contains distinct, meaningful information suitable for inclusion in the KG. Mere presence in the source is insufficient; vacuous fragments, repeated paraphrases, and wrapper nodes with no new information should score below `1`. Textual provenance is checked separately by `substring_valid`.
- `r` / role correctness: whether the assigned role is appropriate for the value in context, for example whether `killed` reasonably serves as an Action or whether a person or entity has an appropriate participant role.
- `m` / minimal value and scope: whether the value span is neither too broad nor too narrow for its role. Where appropriate, this field rewards concise heads or core phrases and penalizes unnecessarily broad or narrow spans.
- `a` / attachment correctness: whether an attribute or child node is attached to the nearest semantically correct parent. This field is normally `null` for ordinary node units.
- `e` / error label: a short diagnostic label such as `wrong_role`, `too_broad`, or `wrong_parent`. It supports error analysis and does not contribute to the numerical score; it is `null` when every applicable `s/r/m/a` field equals `1`.

### Judge-derived metrics

These metrics aggregate the Judge output and measure the semantic precision and structural plausibility of extracted units.

- `judge_coverage`: a run-quality-control metric. It is `1` when the span receives a parseable Judge result and `0` otherwise; it is not a semantic derived metric.
- `correct_unit_yield`: the number of strictly correct units. A unit is counted only when every applicable `s/r/m/a` field equals `1`. This is a yield measure, not a precision measure.
- `strict_unit_precision`: the primary strict unit-level precision metric. A unit scores `1` only when all applicable `s/r/m/a` fields equal `1`; otherwise it scores `0`.
- `role_correctness`: the mean of all applicable `r` fields, measuring the correctness of semantic-role assignment.
- `attachment_correctness`: the mean of all applicable `a` fields for attributes and child links, measuring correct attachment to parent nodes.

### Independent global diagnostics (excluded from the four metrics above)

Global diagnostics use a second independent API call that inspects both the complete `parsed_extraction.components` tree and flattened units with stable IDs. It reports only issue types and affected IDs, produces no overall structural score, and does not modify or feed back into `s/r/m/a`.

- `semantic_redundancy_span_rate`: the proportion of successfully diagnosed spans containing at least one clearly redundant unit, repeated paraphrase, or wrapper node with no new information.
- `unsupported_hierarchy_span_rate`: among eligible spans containing at least one `child_link`, the proportion with a clearly unsupported parent-child relation. Methods with no child links report `N/A`, not `0`.
- Each issue stores its `type`, `anchor_id`, `related_ids`, and a short reason. The deduplication key is fixed as `(span_id, type, anchor_id)`.
- `diagnostic_coverage` is run-level quality control and must be reported with both rates.

### Aggregation

- A text span is the smallest extraction and judging unit: each cause and effect span is extracted and judged separately.
- Span-level metrics are first aggregated to sample-level metrics. A sample may contain several causal relations and therefore several cause and effect spans.
- Method-level metrics are macro-averaged across samples, preventing samples with more causal spans from dominating the result.
- `correct_unit_yield` is summed within each sample and then averaged across samples.
- `None` and `N/A` values are excluded from the corresponding semantic-metric means. Judge failures are reflected by `judge_coverage` and contribute neither semantic-quality nor yield values.
- The four semantic derived metrics are `strict_unit_precision`, `correct_unit_yield`, `role_correctness`, and `attachment_correctness`; `judge_coverage` is quality control only.
- Judge v2 and v3 use the same input format but differ in rubric strictness. Score changes from v2 to v3 should be interpreted as rubric sensitivity, not as direct performance gains or losses on the same scale.


> Judge v2 is retained as a reference. The primary reported results use v3, which was validated on a ten-item paired calibration set. Report scores from different rubric versions separately.


In [ ]:
# ===== KG evaluation: batch evaluation over gold spans =====
# Run this cell after setting RUN_MODE="evaluation":
# - Use only gold causal spans without calling Demo1 causal extraction.
# - Keep local construction extraction sequential.
# - Run the DeepSeek Judge concurrently with JUDGE_MAX_WORKERS workers.
# - Aggregate final results by data point/sample and retain all span-level details for manual inspection.
try:
    from tqdm.auto import tqdm
except Exception:
    class _NoProgress:
        def __init__(self, iterable=None, **_kwargs):
            self.iterable = [] if iterable is None else iterable

        def __iter__(self):
            return iter(self.iterable)

        def update(self, _value):
            return None

        def close(self):
            return None

    def tqdm(iterable=None, **kwargs):
        return _NoProgress(iterable, **kwargs)


if RUN_MODE != "evaluation":
    print("RUN_MODE!='evaluation'; skipping batch KG evaluation.")
else:
    eval_output_dir = PROJECT_ROOT / "results" / "kg_evaluation"
    eval_schema = resolve_eval_schema(EVENT_PROMPT_VERSION, EVAL_SCHEMA)
    eval_max_depth = resolve_eval_max_depth(EVENT_PROMPT_VERSION, eval_schema, EVAL_MAX_DEPTH)
    eval_samples = load_dataset(EVAL_DATASET)
    eval_span_records = load_gold_span_records(
        eval_samples,
        dataset=EVAL_DATASET,
        sample_n=EVAL_SAMPLE_N,
        max_spans=EVAL_MAX_SPANS,
    )
    eval_gold_sample_count = len({record["sample_id"] for record in eval_span_records})

    print(f"Evaluation dataset: {EVAL_DATASET}")
    print(f"Prompt version: {EVENT_PROMPT_VERSION}")
    print(f"Schema: {eval_schema}; max_depth: {eval_max_depth}")
    print(f"Gold samples: {eval_gold_sample_count}")
    print(f"Gold spans: {len(eval_span_records)}")
    if RUN_JUDGE:
        print(f"Judge: prompt={JUDGE_PROMPT_VERSION}; model={JUDGE_MODEL} (thinking disabled); workers={JUDGE_MAX_WORKERS}; retries={JUDGE_RETRY_TIMES}")
    else:
        print("RUN_JUDGE=False: running extraction and deterministic validation only.")

    eval_config = KGEvalConfig(
        dataset=EVAL_DATASET,
        event_prompt_version=EVENT_PROMPT_VERSION,
        sample_n=EVAL_SAMPLE_N,
        max_spans=EVAL_MAX_SPANS,
        schema=EVAL_SCHEMA,
        max_depth=EVAL_MAX_DEPTH,
        extraction_retry_times=EVAL_EXTRACTION_RETRY_TIMES,
        checkpoint_every=EVAL_CHECKPOINT_EVERY,
        output_dir=eval_output_dir,
        run_judge=RUN_JUDGE,
        judge_prompt_version=JUDGE_PROMPT_VERSION,
        judge_api_key_path=PROJECT_ROOT / JUDGE_API_KEY_PATH,
        judge_model=JUDGE_MODEL,
        judge_max_tokens=JUDGE_MAX_TOKENS,
        judge_timeout=JUDGE_TIMEOUT,
        judge_max_workers=JUDGE_MAX_WORKERS,
        judge_retry_times=JUDGE_RETRY_TIMES,
        judge_retry_base_seconds=JUDGE_RETRY_BASE_SECONDS,
        save_outputs=True,
    )
    eval_result = run_kg_evaluation(
        config=eval_config,
        construction_client=client,
        samples=eval_samples,
        progress_factory=tqdm,
    )

    span_results = eval_result.span_results
    sample_results = eval_result.sample_results
    method_metrics = eval_result.method_metrics
    output_paths = eval_result.output_paths

    print("\n===== KG Evaluation Summary =====")
    for key, value in method_metrics.items():
        print(f"{key}: {value}")
    print("\nSaved files:")
    for name, path in output_paths.items():
        print(f"{name}: {path}")


> When reporting results, record both `EVENT_PROMPT_VERSION` and `JUDGE_PROMPT_VERSION`. Changes from v2 to v3 reflect rubric sensitivity; do not compare values from different Judge versions as measurements on the same scale.
